<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 1 — Product discovery and effective AI use

<small>Data Creators › Day 1 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

<a id="how-to-use"></a>

## About this notebook

- Day 1's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [ ]:
# Setup: the demo environment on PATH, DuckDB and pandas for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day1-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day1-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a DuckDB file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 1 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

## Today

Hours are 09:00–15:00 with lunch at 12:00. Breaks mid-morning and mid-afternoon. The day is design: conversations, paper, the whiteboard, and the first generated documents.

| # | Section | What happens |
|---|---|---|
| 1 | **[Welcome, the room and the operating model](#s1)** | Welcome · Activity 1: introductions and pain points · Guest: the DXP operating model · Shareback: products and pain points · Whiteboard: what is a data product? · Break |
| 2 | **[Discovery, justification and Catalyst](#s2)** | Whiteboard: how do you design a data product? · Activity 2: justify a product · Debrief · Whiteboard: lifetime value and adoption · Demo: Catalyst · Activity 3: compare independent PRDs · Lunch |
| 3 | **[AI foundations and effective practice](#s3)** | Whiteboard: the seven ideas · Demo: the interface on the running example · Break |
| 4 | **[Interview, generate and the lifecycle](#s4)** | Demo: Voice Memos to transcript · Activity 4: interview and generate · Debrief: requirements discovery · Reflect: the planted-errors contract · Activity 5: explore the lifecycle · Before learners leave |

<a id="s1"></a>

## 1. Welcome, the room and the operating model

### Welcome — the week, and the people in the room

![The five days and what each leaves behind](../common/diagrams/png/week-map.png)

- Monday designs; Tuesday maps and sees the platform; Wednesday and Thursday build; Friday you build one on your own.
- Everything you make this week is kept and reused the next day.
- Friday's capstone certifies: you build a data product on the paved path and we watch.

### Activity 1 — introductions and pain points

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 1 &nbsp;—&nbsp; Introductions and pain points &nbsp;·&nbsp; Tables &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Five minutes: go round the table with team, role and one hobby, and find a hobby you share. Ten minutes: list the data products you work on or depend on, and what is annoying about each one. Write them on the paper; a note taker keeps the list.</strong><br><br><strong>Produce:</strong><br>• A written list of the table's data products (one line each: name, who uses it, what for)<br>• For each, the sharpest pain point<br>• A group name<br><br><strong>Debrief questions:</strong><br>• Which products does the table rely on but not trust, and why?<br>• Who makes a decision with each one?</div></div>

### Guest — the DXP operating model: the sponsor, or the fallback below

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Can you find the positions of one fund, with confidence it is the right table, in three seconds?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Four personas: consumers, business data owners, platform providers, data product creators.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Technologists have made the data calls for decades without the business context or the policy; that is unfair and unsafe.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Creators sit at the intersection: consumer need, owner priorities, the paved path. Build once, reuse, publish with metadata.</div></div></div>

![The four personas, with the data product creator at the intersection](../common/diagrams/png/d1-personas.png)

- Consumers: everyone who needs data, including you, and increasingly AI.
- Business data owners: accountable per domain; the DXP site lists owners and stewards.
- Platform providers: one way to create data, with standards, catalogs and metadata, instead of choose-your-own-platform.
- Creators: reusable, trusted products that a consumer finds in a catalog instead of asking a person.

<div style="border:2px solid #F4CCCC; border-radius:8px; margin:10px 0;">
<div style="background:#F4CCCC; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🧯 IF IT BREAKS</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">No sponsor today? Give the opening yourself in five minutes, then move on.</div></div>

- The three-second test, the four personas and the publishing promise, from the anchor.
- Log every question about access or policy in the Q&A table for the platform team.

### Shareback — products and pain points

Type them in as the tables report back. This table comes back on Day 2 (which contract to extend) and Day 5 (what to compare Atlas with).

| Table | Product | Who decides with it | Pain point |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

**🖊️ Whiteboard — share-out probes**

- Who makes a decision with this product?
- What makes it untrustworthy?
- Why does alignment cost so much here?

### Whiteboard — what is a data product?

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Is a vendor file dropped in a folder a data product?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A data set plus promises, made of data, code and infrastructure, that realigns source data to a consumer's decision.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>The value is in the realignment and the promises kept; a bare extract is nobody's product.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Promises are written in contracts and enforced by tests, schedules and ownership. Friday you build one.</div></div></div>

**🖊️ Whiteboard — what is a data product**

- What properties does a data product have?
- How are the promises enforced?
- Is Morningstar's file in a folder a data product?
- What does 'aligned' mean?
- What is a data product not?

![Three views of a data product: promises, assets, alignment](../common/diagrams/png/d1-three-views.png)

- The promises view does most of the work this week: SLAs, ownership, quality, change management.
- The assets view is what the platform sees: data, code, infrastructure.
- The alignment view is why anyone pays for it.

![Source-aligned to consumer-aligned, with the promises enforced by code](../common/diagrams/png/d1-alignment.png)

- Source-aligned is their shape; consumer-aligned is the shape of the decision; the domain layer agrees one meaning in between.
- On Friday the Atlas data arrives source-aligned and you publish consumer-aligned reports; the names src_, dom_, rpt_ follow this picture.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 10:30</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s2"></a>

## 2. Discovery, justification and Catalyst

### Whiteboard — how do you design a data product?

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Someone says 'we need a data product'. What do you ask first?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>The triage questions: who needs it and for what, does it exist, can we build it, is it worth it.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Most failed products were buildable; they were not needed, not adopted, or not worth their lifetime cost.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Answer the non-technical questions first, on paper, with the consumer in the room.</div></div></div>

![The triage questions, in the order a creator asks them](../common/diagrams/png/d1-triage.png)

- Who needs it and which decision does it support is the first question, not which platform.
- 'Does it exist?' includes the workaround people run today; that workaround is the product's competitor.
- Can we build it leads to the Day 2 questions: sources, transformations, sensitivity, grain, cadence.
- Worth it: lifetime value against lifetime cost, and whether it deserves priority now.

**🖊️ Whiteboard — how do you design a data product**

- Someone says we need a data product. First questions?
- Suppose yes to all. Now what?
- And then?

### Activity 2 — justify a product

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 2 &nbsp;—&nbsp; Justify a product &nbsp;·&nbsp; Tables &nbsp;·&nbsp; 20 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Pick one product from your list, or propose a new one for the firm. Answer the non-technical questions below; leave 'how would we build it' for later in the week. Note the question that was hardest, most surprising or most interesting.</strong><br><br><strong>Produce:</strong><br>• Who needs it, and what business problem or decision it supports<br>• Whether something like it exists, and what people do today<br>• Whether it can be built, and what would make that hard<br>• Whether the benefit is worth the development and ongoing cost<br>• Why consumers would adopt it, and why it should be prioritised now<br>• One unresolved question<br><br><strong>Finished early?</strong> Take a second product, or test your first against a sceptical business owner: what would they say no to?<br><br><strong>Debrief questions:</strong><br>• What changes for the person using this?<br>• Who is the business owner, by role?<br>• Which question could the table not answer, and who could?</div></div>

### Debrief

**🖊️ Whiteboard — Activity 2 debrief**

- What was the data product, and what was the hardest question?
- What changes for the person using this?
- Globalisation, less mature regions, regulation: what kind of problem is that?

### Whiteboard — lifetime value and adoption

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">How do you estimate the lifetime value of a product?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Costs over the life of the product against the payoffs, most of which are hard to price; adoption decides whether any payoff arrives.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Things near the money get prioritised; far from it you must make the value visible and remove the barriers to adoption.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Elicit costs, payoffs, barriers and responses; leadership makes the call on what cannot be priced.</div></div></div>

![Costs, payoffs, adoption barriers and responses](../common/diagrams/png/d1-value-adoption.png)

- Price consistency by the alignment meetings it removes; price saved time where the consumer is near revenue.
- Most barriers come down to safety: trust in the data, trust that the payoff is real, who is liable if it goes wrong.
- The responses are leadership moves: take the liability, show it working, record the walkthrough, set a deadline, follow through.

**🖊️ Whiteboard — lifetime value and adoption**

- What does a data product cost over its life?
- What are the payoffs?
- Why do data products fail?
- What do we do about it?

### Demo — Catalyst: a skill is a prompt

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Where does a reviewed PRD come from in ten minutes?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Catalyst: CG's repository of AI skills, saved prompts and scripts, for every phase of the data product lifecycle.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Documents and code aligned to CG standards, the same way for everyone; the concierge does everything and is slow, specific skills are fast.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>In Copilot Chat: /ai-catalyst and the skill name, then your description or a transcript. Then review every line.</div></div></div>

![Catalyst's four tiers of skills and the nine lifecycle phases](../common/diagrams/png/d1-catalyst.png)

- Orchestrators call workflow skills which call document and capability skills; open a lifecycle skill file and you can see the calls.
- Document skills produce one artefact from a template: PRD, data contract, transformation spec, runbook.
- Capability skills do one thing and may ship Python: search Databricks, profile a table, write a dbt model, generate a DAG.
- Nine phases, 0 to 8: prereq, triage, discovery, design, build, validate, publish, operate, evolve. Today is discovery and design.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the data-contract skill, then a PRD from the transcript</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">A Catalyst skill is a prompt file: watch one run on the transcript, then find the invented detail.</div></div>

1. Open `.claude/skills/data-contract/SKILL.md` (installed by `common/catalyst/install-skills.sh`; the same file learners see in the Catalyst repo): the eight required sections and the quality checks.
2. Open `.claude/skills/data-contract/templates/data-contract.md`: identity, schema, SLAs, quality rules, access, change management, consumer registry, approvals, revision history.
3. Open `.claude/skills/dplc-design/SKILL.md`: it chains requirements-acceptance-criteria, feasibility-report, consumer-use-case-map and data-model.
4. Open `.claude/skills/dplc-concierge/SKILL.md` and say: start it Friday, maybe a product by Monday.
5. Learners' form: Copilot Chat, `/ai-catalyst product-requirements-document`, paste a description.
6. Instructor's form: Claude Code in `courseware/`, `/product-requirements-document`, paste `common/running-example/interview-transcript.md`.
7. Read one section of the result aloud and find the invented detail before accepting anything.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/ai-catalyst product-requirements-document
A US end-of-day price sheet for the US product team: closing prices in USD for active US-listed instruments, one row per instrument per business day, from the vendor price feed, the instrument master and the FX feed, published by 07:30 ET. Consumers: pricing analysts, the sales desk, risk.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/product-requirements-document  (then paste common/running-example/interview-transcript.md)</pre></div></div>

<div style="border:2px solid #F4CCCC; border-radius:8px; margin:10px 0;">
<div style="background:#F4CCCC; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🧯 IF IT BREAKS</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Catalyst not installed or slow on the demo laptop: review the prepared PRD instead; the lesson survives without the generation.</div></div>

- Open `common/running-example/prd.md`: the reviewed output, with the corrections in brackets.
- Walk the Key Product Information, Problem and Value, and Consumers sections.

### Activity 3 — compare independent PRDs

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 3 &nbsp;—&nbsp; Compare independent PRDs &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 10 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Both of you describe the same product to Catalyst, separately and without showing each other, and generate a PRD each. Then compare scope, consumers, requirements and assumptions. Find one or two differences that matter.</strong><br><br><strong>Produce:</strong><br>• Two PRDs for one product<br>• One or two differences that matter, and which input caused them<br>• One correction or piece of context each of you would add<br><br><strong>Finished early?</strong> Swap one requirement with another pair: is it precise enough to check? Bring the disagreement to the debrief.<br><br><strong>Debrief questions:</strong><br>• Where did the detail of the input change the output?<br>• What did the model invent: a field, an owner, an approval, a revision history?</div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s3"></a>

## 3. AI foundations and effective practice

### Whiteboard — the seven ideas, in order

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Why is the same model brilliant at SQL and unreliable about your fund master?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A model predicts text from compressed training data; a harness turns its tool calls into actions; a skill is a prompt that steers it.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Knowing what the system does lets you choose models and settings, give the right context and review what comes back.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Seven ideas in order, then the interface on a real task: model, effort, context, a fresh conversation.</div></div></div>

![Training makes the model; inference runs it; the context window is everything so far](../common/diagrams/png/d1-ai-foundations.png)

- Pre-training compresses a corpus; instruction tuning and preference tuning make it follow instructions and preferences.
- Code is the one domain where preferences can be checked automatically, which is why generation is strong there.
- Hidden reasoning tokens are what 'thinking effort' buys; more is not always better.
- The prompt is the whole conversation: quality falls as it fills; summarise to a file and start fresh.

![You, the harness, the model and the tools: a tool call is JSON the harness executes](../common/diagrams/png/d1-harness.png)

- Copilot in VS Code and Claude Code are harnesses; Opus, Sonnet and the GPT family are models.
- An agent is a harness plus a model: the model proposes actions, the harness performs them.
- A Catalyst skill is a prompt that pulls the model towards the right tool calls and the right template.

**🖊️ Whiteboard — the seven ideas, in order**

- 1. What is AI? What is intelligence?
- 2. How is a model trained?
- 3. What is inference and reasoning?
- 4. What is a tool call, a harness, an agent, a skill?
- 5. Which model?
- 6. Which settings?
- 7. How do I know it worked?

### Demo — the interface on the running example

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the interface on the running example</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Copilot Chat and Claude Code side by side: model, effort and context, on one real task.</div></div>

1. Open Copilot Chat (what learners see) and Claude Code (this laptop) side by side.
2. Model picker: name this week's models; say which is least compressed and most expensive.
3. Thinking effort: ask the same question at low and at maximum; compare the time and the answer.
4. Context: show the conversation length, then `/clear` in Claude or a new chat in Copilot; say when to do it.
5. The task: "Add realistic SLAs to this contract" with `common/running-example/data-contract.md` attached.
6. Then: "List every number in the SLA section and where it came from."

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, with the contract attached</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">List every numeric threshold, date, owner and version in this document, and for each one say whether it came from the transcript, the PRD, or was assumed.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">Same prompt; attach the file with @ or paste it.</pre></div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 13:55</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s4"></a>

## 4. Interview, generate and the lifecycle

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">What should we build, now that we agree we should build it?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>An interview of the producer and the consumer, recorded and transcribed, turned into a PRD and a contract with Catalyst.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>The documents are only as good as the conversation; the conversation is where the internal knowledge is.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Threes: producer, consumer, creator. Seven questions. Record, transcribe, generate, inspect, correct.</div></div></div>

![Producer, consumer and creator; transcript to PRD to contract to review](../common/diagrams/png/d1-interview.png)

- The producer knows the source; the consumer knows the decision; the creator reconciles them.
- The transcript is the input that stops the model substituting generic knowledge.
- Review is part of the method: the invented detail is always there.

### Demo — Voice Memos to transcript

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Voice Memos to transcript</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">A recording on the laptop becomes the transcript Catalyst reads: free and good enough.</div></div>

1. Open Voice Memos on the Mac; record thirty seconds of a mock interview; pause; Done.
2. Click Transcript to show the text; click off and on again, then Copy transcript.
3. Paste the transcript into the Catalyst prompt.

### Activity 4 — interview and generate

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 4 &nbsp;—&nbsp; Interview and generate &nbsp;·&nbsp; Threes &nbsp;·&nbsp; 30 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Choose a product you know. One of you is the producer (knows the upstream source and its constraints), one the consumer (knows the decision, the desired result and how they work today), one the creator (interviews both and reconciles them). Record the conversation, then give the transcript to Catalyst: generate a PRD, then a data contract. Inspect and correct both.</strong><br><br><strong>Produce:</strong><br>• A recorded, transcribed interview (Voice Memos on a Mac: record, pause, done, Transcript tab, copy; or a Teams meeting transcript)<br>• A PRD generated from the transcript<br>• A data contract generated after it<br>• Two corrections you made and why<br><br><strong>Finished early?</strong> Ask Catalyst to list every field, owner and date in the contract that did not come from the transcript.<br><br><strong>Debrief questions:</strong><br>• Which identifiers, keys, fields or definitions were wrong or missing?<br>• Where did it substitute generic knowledge for your internal source?<br>• Did the interview leave scope open or lead to an unsuitable table?<br>• What authoritative context fixed it?</div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/ai-catalyst product-requirements-document
&lt;paste the transcript&gt;

Then, in the same conversation:
/ai-catalyst data-contract
Use the PRD above and the transcript. Where the transcript does not say, leave the cell empty rather than inventing a value.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/product-requirements-document  &lt;paste transcript&gt;
then
/data-contract  Use the PRD above; leave unknown cells empty.</pre></div></div>

### Debrief — requirements discovery

![Four ways to find out what consumers need, and what each is good for](../common/diagrams/png/d1-discovery-methods.png)

- Interviews are dynamic; people may be polite, confused or wrong about their own practice.
- Observation shows the workaround they actually run: design from where people are, not from technical heaven.
- Surveys scale but are framed in the creator's words; documents and code show what exists.

**🖊️ Whiteboard — Activity 4 debrief**

- What did the generated documents get wrong?
- How else could we discover requirements?
- What are the sections of a PRD?
- And of a data contract? Why is the glossary there and not in the PRD?

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 REFLECT — the planted-errors contract</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Five invented details are planted in this contract; the room gets three minutes to find them.</div></div>

- Open `common/running-example/data-contract-planted-errors.md` on the projector; the answer key is at the end of the file.
- Name the pattern afterwards: identifiers, dates, versions, approvals and confident numbers are where models fill gaps.

### Activity 5 — explore the lifecycle

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 5 &nbsp;—&nbsp; Explore the lifecycle &nbsp;·&nbsp; Individual, then tables &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Ask Catalyst about itself: the phases of the data product lifecycle at CG, what the build phase requires, and which of those requirements have a skill. Ten minutes asking, five minutes for each table to share one correction and the evidence for it.</strong><br><br><strong>Produce:</strong><br>• The nine phases, in order<br>• The build-phase prerequisites (PRD, data contract, data model, transformation spec) and the skill for each<br>• One thing Catalyst got wrong or vague, with the guidance that corrects it<br><br><strong>Debrief questions:</strong><br>• Which phase are today's artefacts in?<br>• What would you ask Catalyst before using a skill tomorrow?</div></div>

<details><summary><strong>Expected answers</strong> (open after the tables have reported)</summary>

- **Nine phases, 0 to 8:** prerequisites, triage, discovery, design, build, validate, publish, operate, evolve.
- **Build phase requires:** a PRD, a data contract, a data model and a transformation spec; all four have document skills (`product-requirements-document`, `data-contract`, `data-model`, `transformation-spec`).
- **Today's artefacts** sit in discovery (PRD) and design (contract). Tomorrow's STTM is the transformation spec's content.
- **Before using a skill tomorrow:** ask Catalyst what the skill consumes (for `setup-dbt-project` and `write-dbt-model`: the contract, the mapping, sample records).

</details>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ Product notes, interview transcript, reviewed PRD, data contract and open questions saved together (one folder per group, named by group).<br>☐ Each learner can name one correction they made to a generated document and why.<br>☐ The list of the room's products is captured in the anchor's table for tomorrow.<br>☐ Anyone whose Catalyst plugin or dev container is not working is on a list with a named next action.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>